# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview

Review available record sets, fields, columns, and their `@id`s. 

**Note**: All entities are referenced by their unique `@id` fields as per Croissant best practices.

In [ ]:
# List record sets, their @id and available fields
record_sets = metadata.record_sets
if not record_sets:
    print('No record sets found. Please check the metadata.')
else:
    for rs in record_sets:
        print(f"Record set: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print(f"  Columns:")
        for col in rs.columns:
            coldesc = col.description if hasattr(col, 'description') else ''
            print(f"    - {col.name} (@id: {col.id}) {coldesc}")
        print('-' * 40)
# Show the full list of record sets & their ids for copy-paste
record_set_ids = [rs.id for rs in record_sets]
print('Available record set @ids:')
for rsid in record_set_ids:
    print('  ', rsid)

## 3. Data Extraction

Load data from each record set into a `pandas.DataFrame`. 

Remember, all record sets and fields should be referenced using their `@id` from above.

In [ ]:
# Prepare to extract all record sets
dataframes = dict()

# List record sets by @id
record_set_ids = [rs.id for rs in metadata.record_sets]

# Download and display top rows for each record set
for rs_id in record_set_ids:
    print(f"--- Loading data for record set: {rs_id} ---")
    records = list(dataset.records(record_set=rs_id))
    # Some record sets may be empty
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for {rs_id}: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found in {rs_id}.")
    print()
# For demonstration, select the main/first record set for EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Main record set selected: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    raise ValueError('No record sets detected in the schema.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalization, grouping. We reference all columns by their column `@id`s.

In [ ]:
# --- Identify numeric and group-by fields from the record set ---
from IPython.display import display

fields = {field.id: field for field in metadata.record_sets[0].fields}

# For demo: find first numeric-type field, and a categorical group field if available
numeric_field_id = None
for fid, field in fields.items():
    if field.data_type in ["Integer", "Float", "Number"]:
        numeric_field_id = fid
        break
# Try to use a grouping field (e.g., 'sex', 'location', 'msi', etc.)
group_field_id = None
for fid, field in fields.items():
    if field.data_type in ["Text", "Boolean"] and fid != numeric_field_id:
        group_field_id = fid
        break
if not numeric_field_id or numeric_field_id not in main_df.columns:
    print('No suitable numeric field found for EDA.')
else:
    # Filter: Example threshold at sample mean for the chosen numeric field
    mean_value = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > mean_value]
    print(f"Filtered records where '{numeric_field_id}' > {mean_value:.2f} :")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by (if available)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print('No suitable group field found for grouping.')

## 5. Visualization

Visualize numeric data distribution and relationship to a categorical/grouping field using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram and boxplot visualization
if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    
    if group_field_id and group_field_id in main_df.columns:
        plt.subplot(1, 2, 2)
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print('Numeric field not found for visualization.')

## 6. Conclusion

- This notebook demonstrated loading, exploring, and visualizing a FAIR^2 clinical cancer dataset using the Croissant schema and `mlcroissant` Python API.
- All dataset entities were referenced using their Croissant `@id` fields, ensuring schema-driven, unambiguous data access and processing.
- Further analysis can refine data cleaning and explore field relationships for clinical research hypotheses.